### 5 · RAG Evaluation Pipeline

You can't improve what you can't measure. This notebook builds a **structured evaluation pipeline**
so every change you make (chunking, retrieval, prompts) can be scored objectively.

### What you'll learn
```
1. How to create a golden evaluation dataset (question / ground_truth / context)
2. LLM-as-judge evaluation — faithfulness, relevance, correctness
3. Using Ragas for automated RAG metrics
4. Building a reusable eval harness you can run after every change
```

### Why this matters
- Production teams **never ship RAG without eval** — it's how you catch regressions
- Tutorials skip this; real projects don't
- This notebook becomes your quality gate before any deployment

In [1]:
import json
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv(override=True)

VECTORSTORE_DIR = Path("../data/vectorstore")
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vectorstore = FAISS.load_local(
    str(VECTORSTORE_DIR), embeddings, allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

print(f"Loaded {vectorstore.index.ntotal} vectors, retriever ready.")

Loaded 292 vectors, retriever ready.


---
### Step 1 — Golden Evaluation Dataset

A **golden dataset** is a hand-crafted set of questions with known correct answers.
This is the most important artifact in any RAG project.

**Rules for good eval questions:**
- Cover different doc types (architecture, features, operations)
- Mix easy (single-doc) and hard (multi-doc reasoning) questions
- Include at least one out-of-domain question
- Write `ground_truth` answers that are factual and verifiable from your docs

In [2]:
# ── Golden Evaluation Dataset ─────────────────────────────────────────────────
# Each entry: question, ground_truth (expected answer), expected_source (for retrieval check)

EVAL_DATASET = [
    {
        "question": "What state management libraries does the AMS Admin Tool use?",
        "ground_truth": "The AMS Admin Tool uses Redux Toolkit for global client-side state, TanStack Query for server data, redux-persist for persisted local state, and useState/useReducer for local UI state.",
        "expected_source": "architecture/redux.md",
    },
    {
        "question": "How does the Journey Canvas render nodes visually?",
        "ground_truth": "The Journey Canvas uses React Flow (@xyflow/react) to render nodes as an interactive flowchart with drag-and-drop, visual edge connections, and auto-layout via dagre.",
        "expected_source": "features/journey-canvas.md",
    },
    {
        "question": "What are the anti-patterns to avoid according to the code quality guide?",
        "ground_truth": "Anti-patterns include storing server data in Redux, dispatching in render causing infinite loops, reading stale closure state in async operations, and mutating state outside Immer reducers.",
        "expected_source": "architecture/code-quality.md",
    },
    {
        "question": "What Firestore collections does the Action Builder use?",
        "ground_truth": "The Action Builder uses environment-scoped collections like actiontype-wip, actiontype-prewip, actiontype-dev, etc.",
        "expected_source": "features/action-builder.md",
    },
    {
        "question": "How does the theming system handle multiple themes?",
        "ground_truth": "The theming system uses Material UI's ThemeProvider with custom theme configurations. Theme preference is persisted via redux-persist.",
        "expected_source": "architecture/theming.md",
    },
    {
        "question": "What is the deployment process for the application?",
        "ground_truth": "The deployment process details are documented in the operations section.",
        "expected_source": "operations/code-quality-audit-plan.md",
    },
    {
        "question": "What is the capital of France?",
        "ground_truth": "This question is out of domain. The documentation does not contain information about geography.",
        "expected_source": None,
    },
]

print(f"Eval dataset: {len(EVAL_DATASET)} questions")
for i, e in enumerate(EVAL_DATASET):
    print(f"  [{i}] {e['question'][:70]}...")

Eval dataset: 7 questions
  [0] What state management libraries does the AMS Admin Tool use?...
  [1] How does the Journey Canvas render nodes visually?...
  [2] What are the anti-patterns to avoid according to the code quality guid...
  [3] What Firestore collections does the Action Builder use?...
  [4] How does the theming system handle multiple themes?...
  [5] What is the deployment process for the application?...
  [6] What is the capital of France?...


---
### Step 2 — Run RAG Pipeline on Eval Dataset

For each question: retrieve → generate → store results.
We collect both the retrieved contexts and the generated answer for scoring.

In [3]:
ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant for the AMS Admin Tool team. "
     "Use the following context to answer the user's question accurately and concisely. "
     "If the context doesn't contain enough information, say so — don't make things up.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

answer_chain = ANSWER_PROMPT | llm | StrOutputParser()


def run_rag(question: str) -> dict:
    """Run retrieval + generation, return docs and answer."""
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join(
        f"[{d.metadata.get('source', '?')}]\n{d.page_content}" for d in docs
    )
    answer = answer_chain.invoke({"question": question, "context": context})
    return {
        "question": question,
        "answer": answer,
        "contexts": [d.page_content for d in docs],
        "source_documents": docs,
    }


# Run all eval questions
eval_results = []
for entry in EVAL_DATASET:
    result = run_rag(entry["question"])
    result["ground_truth"] = entry["ground_truth"]
    result["expected_source"] = entry["expected_source"]
    eval_results.append(result)
    print(f"✓ {entry['question'][:60]}...")

print(f"\nGenerated {len(eval_results)} answers.")

✓ What state management libraries does the AMS Admin Tool use?...
✓ How does the Journey Canvas render nodes visually?...
✓ What are the anti-patterns to avoid according to the code qu...
✓ What Firestore collections does the Action Builder use?...
✓ How does the theming system handle multiple themes?...
✓ What is the deployment process for the application?...
✓ What is the capital of France?...

Generated 7 answers.


---
### Step 3 — Manual LLM-as-Judge Evaluation

Before using a framework, understand what's being measured.
We build **3 custom judges** using the LLM:

| Metric | Question it answers |
|--------|--------------------|
| **Faithfulness** | Is the answer supported by the retrieved context? (no hallucination) |
| **Answer Relevance** | Does the answer actually address the question? |
| **Context Relevance** | Are the retrieved chunks relevant to the question? |

This is exactly how production eval works — LLM judges score each dimension.

In [4]:
# ── Judge 1: Faithfulness ─────────────────────────────────────────────────────
# Does the answer ONLY use information from the context?

FAITHFULNESS_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an evaluation judge. Given a context and an answer, determine if the answer "
     "is FULLY supported by the context. Score 1 if every claim in the answer can be traced "
     "back to the context, 0 if any claim is fabricated or unsupported.\n\n"
     "Respond with ONLY a JSON object: {{\"score\": 0 or 1, \"reason\": \"brief explanation\"}}"),
    ("human",
     "Context:\n{context}\n\nAnswer:\n{answer}"),
])

# ── Judge 2: Answer Relevance ─────────────────────────────────────────────────
# Does the answer address the question?

RELEVANCE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an evaluation judge. Given a question and an answer, determine if the answer "
     "is relevant and addresses the question. Score 1 if the answer directly addresses "
     "the question, 0 if it's off-topic or doesn't answer what was asked.\n\n"
     "Respond with ONLY a JSON object: {{\"score\": 0 or 1, \"reason\": \"brief explanation\"}}"),
    ("human",
     "Question:\n{question}\n\nAnswer:\n{answer}"),
])

# ── Judge 3: Context Relevance ────────────────────────────────────────────────
# Are the retrieved chunks relevant to the question?

CONTEXT_RELEVANCE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an evaluation judge. Given a question and a list of retrieved text chunks, "
     "score how many of the chunks are relevant to answering the question. "
     "Return a score between 0.0 and 1.0 where 1.0 means all chunks are relevant.\n\n"
     "Respond with ONLY a JSON object: {{\"score\": float, \"reason\": \"brief explanation\"}}"),
    ("human",
     "Question:\n{question}\n\nRetrieved chunks:\n{context}"),
])

print("Judge prompts defined: faithfulness, answer_relevance, context_relevance")

Judge prompts defined: faithfulness, answer_relevance, context_relevance


In [5]:
import json as json_mod


def parse_judge_response(text: str) -> dict:
    """Parse JSON from LLM judge response, with fallback."""
    try:
        # Strip markdown code fences if present
        cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json_mod.loads(cleaned)
    except json_mod.JSONDecodeError:
        return {"score": 0, "reason": f"Failed to parse: {text[:100]}"}


def evaluate_single(result: dict) -> dict:
    """Run all 3 judges on a single eval result."""
    context_str = "\n\n---\n\n".join(result["contexts"])

    # Faithfulness
    faith_resp = (FAITHFULNESS_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context_str, "answer": result["answer"]}
    )
    faith = parse_judge_response(faith_resp)

    # Answer Relevance
    rel_resp = (RELEVANCE_PROMPT | llm | StrOutputParser()).invoke(
        {"question": result["question"], "answer": result["answer"]}
    )
    relevance = parse_judge_response(rel_resp)

    # Context Relevance
    ctx_resp = (CONTEXT_RELEVANCE_PROMPT | llm | StrOutputParser()).invoke(
        {"question": result["question"], "context": context_str}
    )
    ctx_rel = parse_judge_response(ctx_resp)

    return {
        "faithfulness": faith,
        "answer_relevance": relevance,
        "context_relevance": ctx_rel,
    }


# Run judges on all results
eval_scores = []
for r in eval_results:
    scores = evaluate_single(r)
    eval_scores.append(scores)
    print(f"✓ {r['question'][:50]}... → F:{scores['faithfulness']['score']} R:{scores['answer_relevance']['score']} C:{scores['context_relevance']['score']}")

print("\nAll evaluations complete.")

✓ What state management libraries does the AMS Admin... → F:1 R:1 C:1.0
✓ How does the Journey Canvas render nodes visually?... → F:1 R:1 C:1.0
✓ What are the anti-patterns to avoid according to t... → F:1 R:1 C:1.0
✓ What Firestore collections does the Action Builder... → F:1 R:1 C:1.0
✓ How does the theming system handle multiple themes... → F:1 R:1 C:1.0
✓ What is the deployment process for the application... → F:1 R:0 C:0.0
✓ What is the capital of France?... → F:1 R:0 C:0.0

All evaluations complete.


In [6]:
# ── Summary Scorecard ─────────────────────────────────────────────────────────

faith_scores = [s["faithfulness"]["score"] for s in eval_scores]
rel_scores = [s["answer_relevance"]["score"] for s in eval_scores]
ctx_scores = [s["context_relevance"]["score"] for s in eval_scores]

print("=" * 70)
print("RAG EVALUATION SCORECARD")
print("=" * 70)
print(f"{'Metric':<25} {'Avg Score':>10} {'Min':>8} {'Max':>8}")
print("-" * 70)
print(f"{'Faithfulness':<25} {sum(faith_scores)/len(faith_scores):>10.2f} {min(faith_scores):>8.2f} {max(faith_scores):>8.2f}")
print(f"{'Answer Relevance':<25} {sum(rel_scores)/len(rel_scores):>10.2f} {min(rel_scores):>8.2f} {max(rel_scores):>8.2f}")
print(f"{'Context Relevance':<25} {sum(ctx_scores)/len(ctx_scores):>10.2f} {min(ctx_scores):>8.2f} {max(ctx_scores):>8.2f}")
print("=" * 70)

# Per-question breakdown
print(f"\n{'Question':<55} {'Faith':>6} {'Rel':>6} {'Ctx':>6}")
print("-" * 75)
for i, (r, s) in enumerate(zip(eval_results, eval_scores)):
    print(f"{r['question'][:53]:<55} {s['faithfulness']['score']:>6} {s['answer_relevance']['score']:>6} {s['context_relevance']['score']:>6}")

RAG EVALUATION SCORECARD
Metric                     Avg Score      Min      Max
----------------------------------------------------------------------
Faithfulness                    1.00     1.00     1.00
Answer Relevance                0.71     0.00     1.00
Context Relevance               0.71     0.00     1.00

Question                                                 Faith    Rel    Ctx
---------------------------------------------------------------------------
What state management libraries does the AMS Admin To        1      1    1.0
How does the Journey Canvas render nodes visually?           1      1    1.0
What are the anti-patterns to avoid according to the         1      1    1.0
What Firestore collections does the Action Builder us        1      1    1.0
How does the theming system handle multiple themes?          1      1    1.0
What is the deployment process for the application?          1      0    0.0
What is the capital of France?                               1      

---
### Step 4 — Retrieval Accuracy (Hit Rate)

Beyond LLM judges, check if the **right document** was retrieved.
This is a simple but powerful metric: did the expected source appear in top-k?

In [7]:
# ── Retrieval Hit Rate ────────────────────────────────────────────────────────

hits = 0
total = 0

print(f"{'Question':<55} {'Expected Source':<40} {'Hit?':>5}")
print("-" * 105)

for r in eval_results:
    expected = r["expected_source"]
    if expected is None:
        continue  # skip out-of-domain

    total += 1
    retrieved_sources = [d.metadata.get("source", "") for d in r["source_documents"]]
    hit = expected in retrieved_sources
    if hit:
        hits += 1

    print(f"{r['question'][:53]:<55} {expected:<40} {'YES' if hit else 'NO':>5}")

print(f"\nRetrieval Hit Rate: {hits}/{total} = {hits/total:.0%}")
print("(Did the expected source document appear in top-5 results?)")

Question                                                Expected Source                           Hit?
---------------------------------------------------------------------------------------------------------
What state management libraries does the AMS Admin To   architecture/redux.md                      YES
How does the Journey Canvas render nodes visually?      features/journey-canvas.md                 YES
What are the anti-patterns to avoid according to the    architecture/code-quality.md               YES
What Firestore collections does the Action Builder us   features/action-builder.md                 YES
How does the theming system handle multiple themes?     architecture/theming.md                    YES
What is the deployment process for the application?     operations/code-quality-audit-plan.md       NO

Retrieval Hit Rate: 5/6 = 83%
(Did the expected source document appear in top-5 results?)


---
### Step 5 — Ragas Automated Evaluation

[Ragas](https://docs.ragas.io/) is the industry-standard framework for RAG evaluation.
It automates exactly what we built manually above, plus more sophisticated metrics.

Key Ragas metrics:
- **Faithfulness** — Is the answer grounded in context?
- **Answer Relevancy** — Does the answer address the question?
- **Context Precision** — Are relevant chunks ranked higher?
- **Context Recall** — Did we retrieve everything needed?

In [8]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Build Ragas dataset from our eval results
samples = []
for r in eval_results:
    samples.append(
        SingleTurnSample(
            user_input=r["question"],
            response=r["answer"],
            retrieved_contexts=r["contexts"],
            reference=r["ground_truth"],
        )
    )

eval_dataset = EvaluationDataset(samples=samples)
print(f"Ragas dataset ready: {len(samples)} samples")

/Users/sauravmajumdar/Developer/AI/move-mind-ai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ragas dataset ready: 7 samples


/var/folders/04/zfbk98sd6bz0fzdyx1bq1hj40000gn/T/ipykernel_59500/1492959592.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/var/folders/04/zfbk98sd6bz0fzdyx1bq1hj40000gn/T/ipykernel_59500/1492959592.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/var/folders/04/zfbk98sd6bz0fzdyx1bq1hj40000gn/T/ipykernel_59500/1492959592.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'raga

In [10]:
# Run Ragas evaluation
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

ragas_result = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

# Extract scores from the pandas dataframe
df = ragas_result.to_pandas()
metric_cols = [c for c in df.columns if c not in ("user_input", "response", "retrieved_contexts", "reference")]

print("\n" + "=" * 70)
print("RAGAS EVALUATION RESULTS")
print("=" * 70)
for col in metric_cols:
    avg = df[col].mean()
    print(f"  {col:<30} {avg:.4f}")
print("=" * 70)

/var/folders/04/zfbk98sd6bz0fzdyx1bq1hj40000gn/T/ipykernel_59500/2338361137.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
/var/folders/04/zfbk98sd6bz0fzdyx1bq1hj40000gn/T/ipykernel_59500/2338361137.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)
Evaluating:   4%|▎         | 1/28 [00:05<02:41,  5.98s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instea


RAGAS EVALUATION RESULTS
  faithfulness                   1.0000
  answer_relevancy               0.5990
  context_precision              0.4968
  context_recall                 0.7857


In [11]:
# Per-question Ragas scores
print(df[["user_input"] + metric_cols].to_string(index=False))

                                                              user_input  faithfulness  answer_relevancy  context_precision  context_recall
            What state management libraries does the AMS Admin Tool use?           1.0          0.788667           0.477778             0.5
                      How does the Journey Canvas render nodes visually?           1.0          0.865064           1.000000             1.0
What are the anti-patterns to avoid according to the code quality guide?           1.0          1.000000           0.500000             1.0
                 What Firestore collections does the Action Builder use?           1.0          0.921936           0.750000             1.0
                     How does the theming system handle multiple themes?           1.0          0.617202           0.750000             1.0
                     What is the deployment process for the application?           1.0          0.000000           0.000000             0.0
                    

---
### Step 6 — Save Eval Results

Persist results so you can compare across experiments.
Every time you change chunking, retrieval, or prompts — re-run this notebook
and compare the scores.

In [12]:
from datetime import datetime

eval_output = {
    "timestamp": datetime.now().isoformat(),
    "config": {
        "embedding_model": EMBEDDING_MODEL,
        "llm_model": LLM_MODEL,
        "retriever_k": 5,
        "num_vectors": vectorstore.index.ntotal,
    },
    "custom_scores": {
        "faithfulness": sum(faith_scores) / len(faith_scores),
        "answer_relevance": sum(rel_scores) / len(rel_scores),
        "context_relevance": sum(ctx_scores) / len(ctx_scores),
        "retrieval_hit_rate": hits / total,
    },
    "per_question": [
        {
            "question": r["question"],
            "answer": r["answer"],
            "ground_truth": r["ground_truth"],
            "scores": s,
        }
        for r, s in zip(eval_results, eval_scores)
    ],
}

output_path = Path("../data/eval_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w") as f:
    json.dump(eval_output, f, indent=2, default=str)

print(f"Eval results saved to {output_path}")
print("Re-run this notebook after every pipeline change to track improvements.")

Eval results saved to ../data/eval_results.json
Re-run this notebook after every pipeline change to track improvements.


---
### Takeaways

| What you learned | Why it matters |
|-----------------|----------------|
| Golden eval dataset | Foundation of all quality measurement |
| LLM-as-judge (custom) | Understand what each metric actually measures |
| Ragas framework | Industry standard, saves time at scale |
| Retrieval hit rate | Simple but catches retrieval regressions fast |
| Saved results | Compare across experiments objectively |

**Production pattern:** Run eval automatically in CI/CD. If scores drop below a threshold, block the deployment.